# 🧠 Laxman AI — Khoj on Google Colab

Free-first bootstrap for the `Kritim` project. This notebook installs Khoj with its local dependencies, starts the server on port `42110`, and exposes it with a temporary Cloudflare Quick Tunnel.

**Important:** Colab is temporary. A runtime restart stops Khoj and its tunnel. Do not put provider API keys in this notebook or GitHub.

In [ ]:
# 1) Install Khoj
!python -m pip install -q --upgrade pip
!python -m pip install -q 'khoj[local]'

In [ ]:
# 2) Prepare persistent-ish working directories in Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib
base = pathlib.Path('/content/drive/MyDrive/LaxmanAI/Khoj')
base.mkdir(parents=True, exist_ok=True)
os.environ['KHOJ_TELEMETRY_DISABLE'] = 'true'
os.environ['USE_EMBEDDED_DB'] = 'true'
print('Khoj workspace:', base)

In [ ]:
# 3) Start Khoj
# Anonymous mode is intended for a private single-user setup.
import subprocess, time, os
log = '/content/khoj.log'
cmd = ['bash', '-lc', 'USE_EMBEDDED_DB=true KHOJ_TELEMETRY_DISABLE=true khoj --anonymous-mode']
proc = subprocess.Popen(cmd, stdout=open(log, 'w'), stderr=subprocess.STDOUT)
time.sleep(8)
print('Khoj process:', proc.poll() if proc.poll() is not None else 'running')
print(open(log, errors='ignore').read()[-5000:])

In [ ]:
# 4) Verify Khoj locally before creating the public tunnel
import requests
try:
    r = requests.get('http://localhost:42110', timeout=10)
    print('Khoj HTTP:', r.status_code, r.url)
except Exception as e:
    print('Khoj is not reachable yet:', repr(e))
    print(open('/content/khoj.log', errors='ignore').read()[-8000:])

In [ ]:
# 5) Install Cloudflare Tunnel and expose port 42110
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /tmp/cloudflared.deb
!dpkg -i /tmp/cloudflared.deb >/dev/null 2>&1 || true
!nohup cloudflared tunnel --url http://localhost:42110 --no-autoupdate >/content/cloudflared.log 2>&1 &
!sleep 8
import re, pathlib
log_text = pathlib.Path('/content/cloudflared.log').read_text(errors='ignore')
urls = re.findall(r'https://[-a-z0-9]+\.trycloudflare\.com', log_text)
print('Temporary Khoj URL:', urls[-1] if urls else 'Not found yet')
print(log_text[-4000:])

## 6) Connect the GitHub Pages frontend

Open `https://laxmannepal.github.io/Kritim/ai/`, press **Connect Khoj**, and paste the `https://…trycloudflare.com` URL printed above.

The temporary URL changes whenever the Colab runtime/tunnel is recreated. Later we will replace it with the stable `ai.laxmannepal.com.np` Cloudflare Tunnel.